In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that, for <code>N</code> three-dimensional points stored on the device, fills <code>indices[i]</code> with the index <code>j ≠ i</code> of the point closest to <code>points[i]</code>. Comparing <em>squared</em> Euclidean distance is sufficient—you do <strong>not</strong> need to compute square-roots.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>External libraries are not permitted</li>
  <li>The final result must be stored in the <code>indices</code> array</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  points  = [(0,0,0), (1,0,0), (5,5,5)]
        indices = [-1, -1, -1]
        N       = 3
Output: indices = [1, 0, 1]   # 0⇆1 are nearest, 2 is closest to 1</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>N</code> ≤ 100,000</li>
  <li>Coordinates are 32-bit floats in the range [-1000, 1000]</li>

  <li>Performance is measured with <code>N</code> = 10,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// points and indices are device pointers
extern "C" void solve(const float* points, int* indices, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# points, indices are tensors on the GPU
@cute.jit
def solve(points: cute.Tensor, indices: cute.Tensor, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# points is a tensor on the GPU
@jax.jit
def solve(points: jax.Array, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.memory import UnsafePointer


# points and indices are device pointers
@export
def solve(
    points: UnsafePointer[Float32, MutExternalOrigin],
    indices: UnsafePointer[Int32, MutExternalOrigin],
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# points and indices are tensors on the GPU
def solve(points: torch.Tensor, indices: torch.Tensor, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# points and indices are tensors on the GPU
def solve(points: torch.Tensor, indices: torch.Tensor, N: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(name="Nearest Neighbor", atol=0, rtol=0, num_gpus=1, access_tier="free")

    def reference_impl(self, points: torch.Tensor, indices: torch.Tensor, N: int):
        """
        Reference implementation that finds the nearest neighbor for each point.
        For N three-dimensional points, fills indices[i] with the index j≠i
        of the point closest to points[i].
        """
        assert points.dtype == torch.float32
        assert indices.dtype == torch.int32
        assert points.shape == (N * 3,)  # N points, each with 3 coordinates
        assert indices.shape == (N,)
        assert N >= 1

        # Reshape points to (N, 3) for easier processing
        pts = points.view(N, 3)

        # pts shape: (N, 3)
        # Expand to (N, 1, 3) and (1, N, 3) for broadcasting
        pts_expand1 = pts.unsqueeze(1)  # (N, 1, 3)
        pts_expand2 = pts.unsqueeze(0)  # (1, N, 3)

        # Compute all pairwise squared distances: (N, N)
        diff = pts_expand1 - pts_expand2  # (N, N, 3)
        dist_sq = torch.sum(diff * diff, dim=2)  # (N, N)

        # Mask diagonal (distance to self) with large value
        mask = torch.eye(N, device=points.device, dtype=torch.bool)
        dist_sq[mask] = float("inf")

        # Find nearest neighbor indices
        indices.copy_(torch.argmin(dist_sq, dim=1).int())

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "points": (ctypes.POINTER(ctypes.c_float), "in"),
            "indices": (ctypes.POINTER(ctypes.c_int), "out"),
            "N": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype_float = torch.float32
        dtype_int = torch.int32
        N = 3

        # Example: points = [(0,0,0), (1,0,0), (5,5,5)]
        points_data = torch.tensor(
            [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 5.0, 5.0, 5.0],  # point 0  # point 1  # point 2
            device="cuda",
            dtype=dtype_float,
        )
        indices_data = torch.full((N,), -1, device="cuda", dtype=dtype_int)

        return {
            "points": points_data,
            "indices": indices_data,
            "N": N,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype_float = torch.float32
        dtype_int = torch.int32
        test_cases = []

        # Test case 1: Basic example from problem description
        test_cases.append(
            {
                "points": torch.tensor(
                    [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 5.0, 5.0, 5.0],  # point 0  # point 1  # point 2
                    device="cuda",
                    dtype=dtype_float,
                ),
                "indices": torch.full((3,), -1, device="cuda", dtype=dtype_int),
                "N": 3,
            }
        )

        # Test case 2: Two points only
        test_cases.append(
            {
                "points": torch.tensor(
                    [0.0, 0.0, 0.0, 3.0, 4.0, 0.0],  # point 0  # point 1
                    device="cuda",
                    dtype=dtype_float,
                ),
                "indices": torch.full((2,), -1, device="cuda", dtype=dtype_int),
                "N": 2,
            }
        )

        # Test case 3: Four points in a square
        test_cases.append(
            {
                "points": torch.tensor(
                    [
                        0.0,
                        0.0,
                        0.0,  # point 0
                        1.0,
                        0.0,
                        0.0,  # point 1
                        0.0,
                        1.0,
                        0.0,  # point 2
                        1.0,
                        1.0,
                        0.0,
                    ],  # point 3
                    device="cuda",
                    dtype=dtype_float,
                ),
                "indices": torch.full((4,), -1, device="cuda", dtype=dtype_int),
                "N": 4,
            }
        )

        # Test case 4: Points with negative coordinates
        test_cases.append(
            {
                "points": torch.tensor(
                    [
                        -1.0,
                        -1.0,
                        -1.0,  # point 0
                        1.0,
                        1.0,
                        1.0,  # point 1
                        0.0,
                        0.0,
                        0.0,  # point 2
                        2.0,
                        2.0,
                        2.0,
                    ],  # point 3
                    device="cuda",
                    dtype=dtype_float,
                ),
                "indices": torch.full((4,), -1, device="cuda", dtype=dtype_int),
                "N": 4,
            }
        )

        # Test case 5: Points with clear unique nearest neighbors
        test_cases.append(
            {
                "points": torch.tensor(
                    [
                        0.0,
                        0.0,
                        0.0,  # point 0
                        10.0,
                        0.0,
                        0.0,  # point 1
                        1.0,
                        0.0,
                        0.0,  # point 2 (closest to 0)
                        11.0,
                        0.0,
                        0.0,  # point 3 (closest to 1)
                        5.0,
                        0.0,
                        0.0,
                    ],  # point 4
                    device="cuda",
                    dtype=dtype_float,
                ),
                "indices": torch.full((5,), -1, device="cuda", dtype=dtype_int),
                "N": 5,
            }
        )

        # Test case 6: Medium random test with fixed seed for reproducibility
        torch.manual_seed(42)
        test_cases.append(
            {
                "points": torch.empty((100, 3), device="cuda", dtype=dtype_float)
                .uniform_(-100.0, 100.0)
                .flatten(),
                "indices": torch.full((100,), -1, device="cuda", dtype=dtype_int),
                "N": 100,
            }
        )

        # Test case 7: Larger test with fixed seed
        torch.manual_seed(123)
        test_cases.append(
            {
                "points": torch.empty((250, 3), device="cuda", dtype=dtype_float)
                .uniform_(-1000.0, 1000.0)
                .flatten(),
                "indices": torch.full((250,), -1, device="cuda", dtype=dtype_int),
                "N": 250,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype_float = torch.float32
        dtype_int = torch.int32
        N = 10000

        return {
            "points": torch.empty((N, 3), device="cuda", dtype=dtype_float)
            .uniform_(-1000.0, 1000.0)
            .flatten(),
            "indices": torch.full((N,), -1, device="cuda", dtype=dtype_int),
            "N": N,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
